<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
      <img src="../../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

In [ ]:
# Common Declarations and setup

import os
import sys
sys.path.insert(0, '../src')
import time
import asyncio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib.gridspec as gridspec

from io import BytesIO

%matplotlib widget

from pathlib import Path
from phaser_functions import *
from phaser_init import init_phaser_sdr

from adi import adf4159
from adi import ad9361
from adi import one_bit_adc_dac
from adi import ad9361
from adi import tddn
from adi.cn0566 import CN0566

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output, HTML 

from dataclasses import dataclass, fields
from typing import List



# Get Script / Notebook root and full path to resources folder
phaser_root = get_phaser_root()
resource_path = phaser_root / "resources"

#display(Markdown(f"Phaser root: **{phaser_root}**"))
#display(Markdown(f"Resource path: **{resource_path}**\n"))

In [ ]:
# Configuration class

@dataclass
class RadarConfig:
    """Configuration parameters for FMCW radar operation."""

    # ========== SDR Parameters ==========
    sample_rate: float = 5.85e6  # NOTE: This is a placeholder! The actual sample rate is calculated
                                  # in configure_sdr() based on buffer_size / frame_time to ensure
                                  # we capture exactly one complete frame (chirp + padding).
                                  # Formula: sample_rate = sdr_buf_size / (ramp_time + pri_padding_ms)
                                  # For 4096 samples / 0.7ms = 5.85 MHz
                                  # ALWAYS use sdr.sample_rate (actual) not config.sample_rate (default) in plots!

    center_freq: float = 2.1e9  # SDR LO frequency (Hz). Upconverted to output_freq by ADF4159
                                 # Keep at 2.1 GHz for optimal Pluto performance

    signal_freq: float = 100e3  # TX baseband tone frequency (Hz). Creates IF offset
                                 # Used to separate DC offset from target returns
                                 # Typical: 100 kHz. Don't change unless you know why

    rx_gain: int = 30  # Receiver gain (dB). Range: -3 to 70 dB
                        # Higher = more sensitive but risk ADC saturation on strong returns
                        # Start at 60, reduce if seeing saturation artifacts
                        # Trade-off: +10 dB gain ~= 3x detection range OR 10 dB less TX power needed

    tx_gain: int = -6  # Transmitter gain (dB). Range: 0 to -88 dB (0 = max power)
                       # 0 dB ~= +30 dBm EIRP with array gain ~= 1W effective
                       # Reduce for short range, regulations, or power saving
                       # Trade-off: -6 dB power ~= 0.5x detection range

    sdr_buf_size: int = 1024 * 8
    fft_size: int = sdr_buf_size

    # ========== Chirp Parameters ==========
    output_freq: float = 9.9e9  # Radar transmit frequency (Hz). X-band (8-12 GHz)
                                 # 9.9 GHz = 30mm wavelength. Good for small targets
                                 # Check local regulations (ISM, amateur, Part 15)

    chirp_BW: float = 500e6  # Chirp bandwidth (Hz). Determines range resolution
                              # Range resolution = c/(2*BW) = 0.3m at 500 MHz
                              # Wider BW = better resolution, more processing
                              # Typical: 250 MHz to 1 GHz (if hardware supports)
                              # Trade-off: 2x BW = 0.5x range resolution (better)

    ramp_time_us: int = 500  # Chirp duration (microseconds). Affects max range
                           # Longer = more samples per chirp = better range resolution
                           # Also affects PRF (pulse repetition frequency)
                           # Typical: 100 to 1000 us
                           # Trade-off: 2x ramp_time_us = 0.5x PRF = 0.5x max unambiguous velocity

    num_chirps: int = 2  # Number of chirps per frame (CPI - Coherent Processing Interval)
                            # More chirps = better Doppler (velocity) resolution
                            # Doppler resolution = lambda/(2*CPI*PRI) where PRI ~= ramp_time_us
                            # Typical: 64 to 512. Power of 2 for efficient FFT
                            # Trade-off: 2x chirps = 0.5x Doppler resolution, 2x processing time

    # ========== Array Parameters ==========
    element_spacing: float = 0.014  # Antenna element spacing (meters). 14mm ~= lambda/2 at 10 GHz
                                     # lambda/2 spacing prevents grating lobes (spatial aliasing)
                                     # Don't change unless physical array changes

    gain_list: List[int] = None  # Per-element gain (0-127). None = all max (127)
                                  # Can apply taper (Blackman, Taylor) to reduce sidelobes
                                  # Example: [8, 34, 84, 127, 127, 84, 34, 8] for Blackman
                                  # Trade-off: Tapering reduces sidelobes but lowers gain

    # ========== Timing Parameters (Advanced) ==========
    begin_offset_fraction: float = 0.05  # Fraction of chirp to skip at start (0.0-0.3)
                                         # VCO takes time to settle; early samples are non-linear
                                         # 0.1 = skip first 10% of chirp (30 us at 300 us ramp)
                                         # Increase if seeing range artifacts near zero
                                         # Trade-off: More offset = fewer samples = less SNR

    pri_padding_ms: float = 0.1   # Dead time between chirps (milliseconds)
                                  # Allows VCO to reset and prevents chirp overlap
                                  # PRI (Pulse Repetition Interval) = ramp_time_us + padding
                                  # Affects PRF and max unambiguous velocity
                                  # Trade-off: More padding = lower PRF = lower max velocity

    # ========== TDD (Time Division Duplex) Parameters ==========
    tdd_trigger_on_raw: int = 0   # TDD GPIO trigger start (raw units)
                                   # Synchronizes chirp generation with data capture
                                   # Keep at 0 for immediate trigger

    tdd_trigger_off_raw: int = 20  # TDD GPIO trigger stop (raw units)
                                    # Pulse width for trigger signal
                                    # Typical: 5-20. Must be long enough for hardware to latch
                                    # Trade-off: Longer pulse more reliable but delays start

    rpi_ip:       str = "192.168.1.10"
    #rpi_ip:       str = "phaser.local"
    sdr_ip:       str = "192.168.2.1"
    fieldfox_ip:  str = "192.168.1.30"

    def __post_init__(self):
        """Set default gain list and calibration file paths if not provided."""
        if self.gain_list is None:
            self.gain_list = [127] * 8

    def __iter__(self):
        for field in fields(self):
            yield field.name, getattr(self, field.name)

#  Default are stored in a dataclass
config = RadarConfig()

#display(Markdown("#### Config values"))
#for name, value in config:    
#    display(Markdown(f"{name} = {value}"))

pll    = None      
gpio   = None
tdd    = None
phaser = None
sdr    = None


In [ ]:
def connect_devices():
    md = """ """
    try:
        global pll, gpio, tdd, phaser, sdr
        display(Markdown(f"- ADF4159: ip: {config.rpi_ip}"))
        pll    = adf4159        (uri="ip:" + config.rpi_ip)
        
        display(Markdown(f"- GPIO: ip: {config.sdr_ip}"))
        gpio   = one_bit_adc_dac(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- TDDN: ip: {config.sdr_ip}"))
        tdd = tddn(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- Phaser: ip: {config.rpi_ip}"))
        phaser = CN0566         (uri="ip:" + config.rpi_ip)

        display(Markdown(f"- SDR: ip: {config.sdr_ip}"))
        sdr    = ad9361         (uri="ip:" + config.sdr_ip)
    
    except:
        display(Markdown(f"Unable to connect to CN0566 / ADF4159. Please check the IP addresses and connections."))
        print("")
        sys.exit(1)

    phaser.sdr = sdr
    
def configure_phaser():
    phaser.configure(device_mode="rx")
    phaser.element_spacing = config.element_spacing
    
    for i in range(0, 8):
        phaser.set_chan_phase(i, 0)

    display(Markdown(f"- Phaser channel phase  = 0"))
    
    for i in range(0, len(config.gain_list)):
        phaser.set_chan_gain(i, config.gain_list[i], apply_cal=False)
    
    phaser._gpios.gpio_tx_sw = 0
    phaser._gpios.gpio_vctrl_1 = 1
    phaser._gpios.gpio_vctrl_2 = 1

def configure_sdr():   
    destroy_sdr_buffer()
    
    # Configure sample rate to capture the full frame (chirp + padding) x no of chirps
    # 
    frame_time_s = ( (config.ramp_time_us * 1e-6) + (config.pri_padding_ms * 1e-3) ) * config.num_chirps

    sr = int(config.sdr_buf_size / frame_time_s)
    phaser.sdr.sample_rate = sr
    
    phaser.sdr.rx_lo = int(config.center_freq)
    phaser.sdr.rx_enabled_channels = [0, 1]
    phaser.sdr.rx_buffer_size = config.sdr_buf_size
    
    phaser.sdr.gain_control_mode_chan0 = 'manual'
    phaser.sdr.gain_control_mode_chan1 = 'manual'
    phaser.sdr.rx_hardwaregain_chan0 = config.rx_gain
    phaser.sdr.rx_hardwaregain_chan1 = config.rx_gain

    phaser.sdr.tx_buffer_size = config.sdr_buf_size
    phaser.sdr.tx_lo = int(config.center_freq)
    phaser.sdr.tx_enabled_channels = [0, 1]
    phaser.sdr.tx_cyclic_buffer = True
    phaser.sdr.tx_hardwaregain_chan0 = -88
    phaser.sdr.tx_hardwaregain_chan1 = int(config.tx_gain)

    display(Markdown(f"""- Setting sample  rate: {sr/1e6:.2f} MHz
- Actual Sample rate: {sdr.sample_rate/1e6:.2f} MHz
- Frame time: {frame_time_s*1e3:.2f} ms (chirp + padding) * Num Chirps
- Chirp time: {config.ramp_time_us} us
- Padding time: {config.pri_padding_ms} ms
- RX LO: {sdr.rx_lo/1e9:.1f} GHz
- TX LO: {sdr.tx_lo/1e9:.1f} GHz
- Buffer size: {sdr.rx_buffer_size} samples
- Capture time: {sdr.rx_buffer_size/sdr.sample_rate*1e3:.2f} ms"""))

def destroy_sdr_buffer():
    try: sdr.tx_destroy_buffer()
    except: pass
    try: sdr.rx_destroy_buffer()
    except: pass

def configure_adf4159():
    # Configure ADF4159 for triggered sawtooth chirp

    vco_freq = int(config.output_freq + config.signal_freq + config.center_freq)
    BW = config.chirp_BW
    num_steps = int(config.ramp_time_us)

    phaser.frequency = int(vco_freq / 4)
    phaser.freq_dev_range = int(BW / 4)
    phaser.freq_dev_step = int((BW / 4) / num_steps)
    phaser.freq_dev_time = int(config.ramp_time_us)
    
    phaser.delay_word = 4095
    phaser.delay_clk = "PFD"
    phaser.delay_start_en = 0
    phaser.ramp_delay_en = 0
    phaser.trig_delay_en = 0
    phaser.ramp_mode = "single_sawtooth_burst"
    phaser.sing_ful_tri = 0
    phaser.tx_trig_en = 1
    phaser.enable = 0                                  # 0 = PLL enable.  Write this last to update all the registers

    display(Markdown(f"""- VCO Output Frequency = {vco_freq/1e9} GHz
- Chirp BW: {(4 * phaser.freq_dev_range)/1e6:.0f} MHz
- Ramp time: {config.ramp_time_us} us
- Chirp rate: {config.chirp_BW/(config.ramp_time_us*1e-6)/1e12:.2f} THz/s"""))

def configure_tdd():
    """
    Configure the TDD (Time Division Duplex) controller so that each FMCW chirp
    is synchronised to the data acquisition hardware.

    The TDD engine generates trigger signals at the start of each chirp period
    (PRI) and repeats this for the required number of chirps in a burst.
    """

    # Route the TDD trigger to the external sync circuitry and enable
    # the Phaser board trigger path.
    gpio.gpio_tdd_ext_sync = True
    gpio.gpio_phaser_enable = True

    # Disable the TDD engine while its configuration is updated.
    tdd.enable = False

    # Use an external trigger source to start the TDD sequence.
    tdd.sync_external = True

    # Begin generating triggers immediately after synchronisation.
    tdd.startup_delay_ms = 0

    # Calculate the Pulse Repetition Interval (PRI).
    #
    # The PRI consists of:
    #   - the FMCW ramp (chirp) duration
    #   - additional padding time between chirps
    #
    # ramp_time_us is stored in microseconds, so convert to milliseconds
    # before adding the padding value.
    PRI_ms = (config.ramp_time_us / 1e3) + config.pri_padding_ms

    
    # Set the time between successive chirp triggers.
    tdd.frame_length_ms = PRI_ms
    
    # Generate one trigger event per chirp in the burst.
    tdd.burst_count = config.num_chirps

    tdd.channel[0].enable = True
    tdd.channel[0].polarity = False
    tdd.channel[0].on_raw = config.tdd_trigger_on_raw
    tdd.channel[0].off_raw = config.tdd_trigger_off_raw

    tdd.channel[1].enable = True
    tdd.channel[1].polarity = False
    tdd.channel[1].on_raw = config.tdd_trigger_on_raw
    tdd.channel[1].off_raw = config.tdd_trigger_off_raw
    
    tdd.channel[2].enable = True
    tdd.channel[2].polarity = False
    tdd.channel[2].on_raw = config.tdd_trigger_on_raw
    tdd.channel[2].off_raw = config.tdd_trigger_off_raw
    
    # Apply the configuration and start the TDD engine.
    tdd.enable = True

    display(Markdown(f"""- Frame Len =  {tdd.frame_length_ms} ms
- No. Chirps / trigger =  {config.num_chirps}"""))

def tx_baseband():
    # Generate baseband transmit waveform (tone at IF)

    fs = int(sdr.sample_rate)
    N = config.sdr_buf_size
    t = np.arange(N) / fs  # FIXED: Guarantees exactly N samples
    
    i_data = np.cos(2 * np.pi * t * config.signal_freq) * 2**14
    q_data = np.sin(2 * np.pi * t * config.signal_freq) * 2**14
    iq_data = i_data + 1j * q_data

    # Validate buffer length BEFORE sending to SDR
    expected_tx_buffer = sdr.tx_buffer_size
    actual_samples = len(iq_data)
    
    
    
    if actual_samples != expected_tx_buffer:
        raise ValueError(f"❌ Buffer length mismatch! Expected {expected_tx_buffer}, got {actual_samples}")
    
    # Send waveform to TX buffer
    sdr.tx([iq_data, iq_data])
    display(Markdown(f"""- Transmit {config.signal_freq/1e3} kHz
- TX buffer size: {expected_tx_buffer} (expected) vs {actual_samples} (actual)
- TX buffer loaded successfully"""))

# FMCW RADAR: Range-Doppler Processing

## Overview

In the previous notebook, we successfully measured the **range** of static targets using synchronized FMCW chirps and the Range FFT.
Range is useful, but so is **velocity** and **bearing**.

In this notebook we will focus on measuring velocity.

## Learning Objectives

By the end of this notebook, you will:
1. Understand the concept of **fast-time** vs **slow-time** in FMCW RADAR
2. Capture a **Coherent Processing Interval (CPI)** of multiple chirps
3. Implement the **2D FFT** for Range-Doppler processing
4. Generate and interpret **Range-Doppler Maps (RDM)**
5. Extract both range and velocity from moving targets


# 1: Velocity in Radar Systems

In radar systems, velocity refers to radial velocity: the component of a target's motion along the radar's line of sight. A target moving directly towards or away from the radar produces a measurable Doppler shift, while a target moving perpendicular to the radar may exhibit little or no Doppler shift. Consequently, Doppler processing measures how quickly a target's range is changing, rather than its complete three-dimensional velocity vector.  

$
\large
v_r = -\frac{dR}{dt}
$

where:

R = target range  
$v_r$ = radial velocity, defined here as positive when approaching and negative when receding

## Why is this important?

When we move from Range FFT to Doppler processing, we're not measuring:

    "How fast is the target moving through space?"

We're measuring:

    "How fast is the target moving towards or away from the radar?"

The Doppler FFT estimates this radial velocity by observing the phase progression of the target return across multiple chirps.

## Can we use the FFT Range Data to Measure Velocity

If radial velocity can be obtained from the rate of change of range, couldn't we estimate it by tracking how the range peak moves between chirps?
The answer is yes, but with some important limitations.

Let's use the FMCW configuration from the previous notebook:

- Ramp duration $T_{\mathrm{ramp}} = 500~\mu\mathrm{s}$
- Padding $T_{\mathrm{pad}} = 100~\mu\mathrm{s}$

To measure velocity, we need to compare multiple chirps. The **Pulse Repetition Interval** (PRI), measured between chirp starts, is
$T_{\mathrm{PRI}} = T_{\mathrm{ramp}} + T_{\mathrm{pad}} = 600~\mu\mathrm{s}$.

Now consider a target travelling at 30 m/s.
The magnitude of the range change between two consecutive chirps is:

$
\large
\begin{aligned} \\
|\Delta R| &= |v_r|T_{\mathrm{PRI}} \\ 
&= 30 \times 0.6\text{ ms} \\ 
&= 18\text{ mm} 
\end{aligned}
$

Recall that our range resolution was approximately:

$
\large
\begin{aligned} \\
\delta R &= \frac{c}{2B} \\
         &= \frac{3\times10^8}{2 \times 500\times10^6} \\
         &= 0.3m
\end{aligned}
$  

Comparing the two values:

Target movement per chirp = 18 mm
Range resolution = 300 mm

The target moves only about 6% of the ideal full-bandwidth range resolution between chirps.
This 30 m/s example illustrates displacement; it exceeds the approximately
12.63 m/s unambiguous velocity limit derived later for this PRI.

As a result, the range peak appears almost stationary in the FFT magnitude plot, making velocity estimation from peak movement both difficult and noisy.

### Could We Improve This?

A few intuitive options come to mind:

#### Increase PRI

This would give the target more time to move between chirps, making changes in range easier to observe.
However, this comes at a cost:

- Slower update rate
- Longer frame times
- Reduced responsiveness to changing targets
- A lower maximum unambiguous Doppler velocity

#### Increase the chirp Bandwidth

Increasing bandwidth improves range resolution:

However:
- Large bandwidths may not be available
- Higher bandwidth often increases hardware cost and complexity
- We'd still be relying solely on changes in FFT magnitude

#### There's a Better Way 

So far, we've only considered the magnitude of the Range FFT.  
However, every FFT bin also contains phase information.  
While magnitude indicates signal strength in a range bin, changes in phase can
reveal small changes in target distance. Absolute phase is periodic and includes
reflection and hardware phase; it does not uniquely locate a target within a bin.  

A target may move only a few millimetres, producing almost no visible change in FFT magnitude, yet that same movement can produce a measurable phase change between consecutive chirps.

By analysing how this phase evolves over time, we can estimate velocity far more accurately than by tracking movement of the range peak alone.

This insight forms the foundation of Doppler processing.

# 2: Doppler Frequency and Velocity Theory

## Doppler Effect Refresher

We use the same sign convention throughout the theory: radial velocity is
**positive for an approaching target** and **negative for a receding target**:

$$
v_r = -\frac{dR}{dt}, \qquad
\lambda = \frac{c}{f_c}, \qquad
f_D = \frac{2v_r}{\lambda} = \frac{2v_r f_c}{c}.
$$

Here $R$ is target range, $f_c$ is carrier frequency, and $c$ is the speed of light.
The factor of two comes from the outward and return propagation paths.

## Ramp Duration and Chirp Repetition Interval

The ramp duration determines the chirp slope. The interval between chirp starts
determines how we sample Doppler:

$$
S = \frac{B}{T_{\mathrm{ramp}}}, \qquad
T_{\mathrm{PRI}} = T_{\mathrm{ramp}} + T_{\mathrm{pad}}, \qquad
\mathrm{PRF} = \frac{1}{T_{\mathrm{PRI}}}.
$$

For this demo, the ramp lasts 500 microseconds and the padding lasts
100 microseconds, giving a **600 microsecond PRI**. The Play-widget interval
controls display updates; it is not the PRI.

## FMCW Doppler: Phase Change Across Chirps

For a coherent target at a fixed range bin, assume the complex signal convention
in which positive Doppler produces increasing phase. After removing a stable
systematic phase offset, the phase change between successive chirps is

$$
\Delta\phi = 2\pi f_D T_{\mathrm{PRI}}, \qquad
v_r = \frac{\lambda\,\Delta\phi}{4\pi T_{\mathrm{PRI}}}.
$$

Use **PRI, including padding**, not just ramp duration. A stationary target has
zero inter-chirp phase change after calibration. Constant velocity produces a
constant inter-chirp phase difference and a steadily advancing accumulated phase.

The receiver's mixer and I/Q convention may reverse the measured phase sign.
The code currently reports positive velocity for increasing corrected phase;
verify which physical direction that represents by moving a reflector toward the
radar. If approaching motion is negative, a sign reversal is needed before labelling
the measured output with the approaching-positive convention used in this theory.
A fixed phase-offset calibration does not determine this direction sign.


## Doppler Measurement Demo 

Lets run the previous demo, were we looked a the FFT range plot.
We will remove the spectrogram and add a plot showing the phaser difference two consecutive plots

We're going to use the TDD engine to help with timing again. You may have noticed:

`tdd.burst_count = 1`I

Setting this property gives us the number of chirps that will be generated by TDD system it will ensure all timings are adhered too.

Lets set `tdd.burst_count = 2`.

We will calculated the avergare of the magnitude and the difference in the phases. 

We should be able to show the phase differences increases/decreases in proportion to the velocity of the target.


In [ ]:
# Hardware Configuration

def hardware_configuration(connect = True):
    if connect is True:
        display(Markdown("**Connect to Devices**"))
        connect_devices()
    
    display(Markdown("**Configure Phaser**"))
    configure_phaser()
    
    display(Markdown("**Configure PlutoSDR**"))
    destroy_sdr_buffer()
    configure_sdr()
    
    display(Markdown("**Configure ADF4159**"))
    configure_adf4159()
    
    display(Markdown("**Configure TDD Engine**"))
    configure_tdd()
    
    display(Markdown("**Transmit Baseband Signal**"))
    tx_baseband()

hardware_configuration()


In [ ]:
# Capture and extraction

def capture_data(phaser, sdr):
    """Trigger TDD and capture synchronized data."""
    phaser._gpios.gpio_burst = 0
    phaser._gpios.gpio_burst = 1
    phaser._gpios.gpio_burst = 0
    time.sleep(0.001)
    return sdr.rx()

def combine_rx_channels(raw_data):
    """Combine both RX channels."""
    return raw_data[0] + raw_data[1]

def extract_chirps(rx_data, num_chirps, sample_rate, config):
    """Extract individual chirps from received buffer."""
    actual_sample_rate = sample_rate

    # Calculate expected samples per chirp period (PRI = ramp + padding)
    PRI_time_s = (config.ramp_time_us * 1e-6) + (config.pri_padding_ms * 1e-3)
    samples_per_PRI = int(actual_sample_rate * PRI_time_s)
    total_samples_needed = samples_per_PRI * num_chirps
    
    # Calculate samples in just the ramp (no padding)
    chirp_samples = int(actual_sample_rate * config.ramp_time_us * 1e-6)
    
    # Validation check
    buffer_size = len(rx_data)
    if buffer_size < total_samples_needed:
        raise ValueError(
            f"Buffer too small for {num_chirps} chirps!"
            f"  Buffer size: {buffer_size} samples"
            f"  Needed: {total_samples_needed} samples"
            f"  Per chirp (PRI): {samples_per_PRI} samples ({PRI_time_s*1e3:.2f} ms)"
            f"  Sample rate: {actual_sample_rate/1e6:.2f} MHz"
            f"  Solution: Increase config.sdr_buf_size to at least {total_samples_needed}"
        )
    
    # Divide buffer into num_chirps sections
    samples_per_section = buffer_size // num_chirps
    
    # Make sure chirp_samples doesn't exceed section size
    chirp_samples = min(chirp_samples, samples_per_section)
    
    # Extract the first chirp_samples from each section
    chirps = []
    for i in range(num_chirps):
        start = i * samples_per_section
        end = start + chirp_samples
        chirp = rx_data[start:end]
        chirps.append(chirp)
    
    return np.array(chirps)

def process_chirps(chirps_2d, return_complex=False):
    """Apply a Blackman window and the fast-time FFT to each chirp.

    The two-chirp demo uses magnitude and phase of the positive half-spectrum.
    The range-Doppler demo requests complex FFT values for its slow-time FFT.
    """
    window = np.blackman(chirps_2d.shape[1])
    spectra = np.fft.fft(chirps_2d * window[None, :], axis=1)
    if return_complex:
        return spectra
    spectra = spectra[:, :spectra.shape[1] // 2]
    return np.abs(spectra), np.angle(spectra)


# Frame processing

@dataclass
class RadarFrame:
    """Processed arrays and target measurements for one capture."""
    rx_data: np.ndarray
    freqs_khz: np.ndarray
    magnitude_db: np.ndarray
    phase_diff_deg: np.ndarray
    range_bins_m: np.ndarray
    target_bin: int
    peak_freq: float
    target_phase_diff_deg: float
    fd: float
    velocity: float

def process_frame(raw_data, sample_rate, config, phase_calibration_deg):
    """Process one capture without accessing hardware or updating the display.

    Retains the existing extraction, FFT indexing, phase averaging and peak
    selection so mathematical corrections can be evaluated separately.
    """
    rx_data = combine_rx_channels(raw_data)
    chirps = extract_chirps(rx_data, config.num_chirps, sample_rate, config)
    magnitudes, phases = process_chirps(chirps)

    actual_sample_rate = sample_rate
    avg_magnitude = np.mean(magnitudes, axis=0)
    magnitude_db = 20 * np.log10(avg_magnitude + 1e-12)

    fft_size = len(avg_magnitude) * 2
    freqs_hz = np.fft.fftfreq(fft_size, 1/actual_sample_rate)[:len(avg_magnitude)]
    freqs_khz = freqs_hz / 1e3

    chirp_time_s = config.ramp_time_us * 1e-6
    chirp_rate = config.chirp_BW / chirp_time_s
    IF_freq = config.signal_freq
    beat_freq = np.abs(freqs_hz - IF_freq)
    range_bins_m = (3e8 * beat_freq) / (2 * chirp_rate)

    target_bin = np.argmax(magnitude_db)

    phase_diff_rad = np.diff(phases, axis=0).mean(axis=0)
    phase_diff_rad = np.arctan2(np.sin(phase_diff_rad), np.cos(phase_diff_rad))
    phase_diff_deg = np.degrees(phase_diff_rad)

    phase_diff_deg = phase_diff_deg + phase_calibration_deg
    phase_diff_deg = np.arctan2(np.sin(np.radians(phase_diff_deg)), 
                                  np.cos(np.radians(phase_diff_deg)))
    phase_diff_deg = np.degrees(phase_diff_deg)

    target_phase_diff_rad = np.radians(phase_diff_deg[target_bin])
    target_phase_diff_deg = phase_diff_deg[target_bin]

    peak_freq = freqs_khz[target_bin]

    TPRI = (config.ramp_time_us * 1e-6) + (config.pri_padding_ms * 1e-3)
    wavelength = (3e8 / config.output_freq)
    fd = target_phase_diff_rad / (2 * np.pi * TPRI)
    velocity = wavelength * fd / 2
    
    return RadarFrame(
        rx_data=rx_data,
        freqs_khz=freqs_khz,
        magnitude_db=magnitude_db,
        phase_diff_deg=phase_diff_deg,
        range_bins_m=range_bins_m,
        target_bin=target_bin,
        peak_freq=peak_freq,
        target_phase_diff_deg=target_phase_diff_deg,
        fd=fd,
        velocity=velocity,
    )

# Plot setup and updates

def render_plot(canvas):
    """Render the figure as PNG bytes for the image widget."""
    with BytesIO() as buffer:
        canvas.print_png(buffer)
        return buffer.getvalue()


def format_target_info(frame, frame_count):
    """Format the live target measurements below the plot."""
    target_range = frame.range_bins_m[frame.target_bin] - CABLE_CALIBARTION_M
    return f"""
    <div style="background:#f8fafc;color:#172b4d;border:1px solid #dbe3ed;
                border-radius:12px;padding:18px 20px;font-family:system-ui,sans-serif;
                font-variant-numeric:tabular-nums;line-height:1.5;">
      <div style="display:flex;justify-content:space-between;align-items:center;
                  flex-wrap:wrap;gap:8px;margin-bottom:14px;">
        <strong style="font-size:16px;">Target measurements</strong>
        <span style="font-size:12px;color:#52637a;background:#e8eef6;
                     border-radius:12px;padding:3px 10px;">Frame {frame_count:,}</span>
      </div>
      <div style="display:flex;flex-wrap:wrap;gap:12px;margin-bottom:14px;">
        <div style="flex:1;min-width:180px;background:#fff;border:1px solid #dbe3ed;
                    border-left:4px solid #2563eb;border-radius:8px;padding:12px 16px;">
          <div style="font-size:12px;color:#52637a;">Target range (corrected)</div>
          <strong style="font-size:28px;color:#1d4ed8;">{target_range:.2f}</strong>
          <span style="font-size:14px;color:#52637a;"> m</span>
        </div>
        <div style="flex:1;min-width:180px;background:#fff;border:1px solid #dbe3ed;
                    border-left:4px solid #0f766e;border-radius:8px;padding:12px 16px;">
          <div style="font-size:12px;color:#52637a;">Radial velocity</div>
          <strong style="font-size:28px;color:#0f766e;">{frame.velocity:+.2f}</strong>
          <span style="font-size:14px;color:#52637a;"> m/s</span>
        </div>
      </div>
      <div style="display:flex;flex-wrap:wrap;gap:12px 28px;font-size:13px;">
        <div><span style="color:#52637a;">Peak frequency</span><br>
             <strong>{frame.peak_freq:.2f} kHz</strong>
             <span style="color:#52637a;"> &middot; bin {frame.target_bin}</span></div>
        <div><span style="color:#52637a;">Phase difference</span><br>
             <strong>{frame.target_phase_diff_deg:+.1f}&deg;</strong></div>
        <div><span style="color:#52637a;">Doppler shift</span><br>
             <strong>{frame.fd:+.1f} Hz</strong></div>
      </div>
    </div>
    """


def create_plots(sample_rate, buffer_size, pri_ms, config):
    """Create the three plots once; return their artists in a dictionary."""
    # Render PNG frames independently of the notebook's interactive backend.
    fig = Figure(figsize=(14, 14), dpi=90)
    canvas = FigureCanvasAgg(fig)
    ax_raw, ax1, ax2 = fig.subplots(3, 1)

    # Expected timing only: sample zero is assumed to align with the first trigger.
    # Use the TDD period readback; the ramp duration is the configured value.
    raw_sample_rate = float(sample_rate)
    expected_pri_us = float(pri_ms) * 1e3
    expected_ramp_us = float(config.ramp_time_us)
    settling_us = config.begin_offset_fraction * expected_ramp_us
    line_raw_i, = ax_raw.plot([], [], color='tab:blue', linewidth=0.6, alpha=0.7, label='I (RX0 + RX1)')
    line_raw_q, = ax_raw.plot([], [], color='tab:orange', linewidth=0.6, alpha=0.7, label='Q (RX0 + RX1)')
    line_raw_abs, = ax_raw.plot([], [], color='black', linewidth=0.9, label='Magnitude')
    for chirp_index in range(config.num_chirps):
        start_us = chirp_index * expected_pri_us
        end_us = start_us + expected_ramp_us
        ax_raw.axvline(start_us, color='tab:green', linestyle='--', linewidth=1.2,
                       label='Expected ramp start' if chirp_index == 0 else None)
        ax_raw.axvline(end_us, color='tab:red', linestyle='--', linewidth=1.2,
                       label='Expected ramp end' if chirp_index == 0 else None)
        ax_raw.axvspan(start_us, start_us + settling_us, color='gold', alpha=0.2,
                       label='Configured settling interval' if chirp_index == 0 else None)
        ax_raw.axvspan(end_us, start_us + expected_pri_us, color='grey', alpha=0.15,
                       label='Expected padding' if chirp_index == 0 else None)
        ax_raw.text(start_us + expected_ramp_us / 2, 0.97, f'Chirp {chirp_index + 1}',
                    transform=ax_raw.get_xaxis_transform(), ha='center', va='top', fontsize=9)
    ax_raw.set_xlabel('Time from first RX sample (us)', fontsize=11)
    ax_raw.set_ylabel('Raw amplitude (ADC counts)', fontsize=11)
    ax_raw.set_title('Raw dechirped I/Q - assuming first trigger at t = 0', fontsize=13, fontweight='bold')
    ax_raw.set_xlim(0, buffer_size / raw_sample_rate * 1e6)
    ax_raw.grid(True, alpha=0.3)
    ax_raw.legend(loc='lower left', fontsize=8, ncol=4)


    line_mag, = ax1.plot([60, 80, 100, 120, 140], [-80, -60, -40, -60, -80], 'b-', linewidth=1)
    peak_marker1, = ax1.plot([100], [-40], 'ro', markersize=10)
    line_phase, = ax2.plot([60, 80, 100, 120, 140], [-100, -50, 0, 50, 100], 'r-', linewidth=1.5)
    peak_marker2, = ax2.plot([100], [0], 'ro', markersize=10)

    peak_range_label = ax1.annotate(
        '', xy=(100, -40), xytext=(12, -12), textcoords='offset points',
        fontsize=10, fontweight='bold', color='darkred', va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red', alpha=0.9),
        annotation_clip=True
    )
    peak_velocity_label = ax2.annotate(
        '', xy=(100, 0), xytext=(12, 12), textcoords='offset points',
        fontsize=10, fontweight='bold', color='darkred', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red', alpha=0.9),
        annotation_clip=True
    )

    ax1.set_xlabel('Beat Frequency (kHz)', fontsize=11)
    ax1.set_ylabel('Magnitude (dB)', fontsize=11)
    ax1.set_title('FFT Spectrum', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim([100, 150])
    ax1.set_ylim([-100, 0])

    ax2.set_xlabel('Beat Frequency (kHz)', fontsize=11)
    ax2.set_ylabel('Phase Difference (degrees)', fontsize=11)
    ax2.set_title('Phase Difference Between Chirps', fontsize=13, fontweight='bold')
    ax2.set_ylim([-10, 10])
    ax2.set_xlim([100, 150])
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()


    return {
        "fig": fig,
        "canvas": canvas,
        "ax_raw": ax_raw,
        "ax1": ax1,
        "ax2": ax2,
        "raw_sample_rate": raw_sample_rate,
        "line_raw_i": line_raw_i,
        "line_raw_q": line_raw_q,
        "line_raw_abs": line_raw_abs,
        "line_mag": line_mag,
        "line_phase": line_phase,
        "peak_marker1": peak_marker1,
        "peak_marker2": peak_marker2,
        "peak_range_label": peak_range_label,
        "peak_velocity_label": peak_velocity_label,
    }


def update_plots(plots, frame, frame_count):
    """Update the existing plots and peak labels from one processed frame."""
    raw_time_us = np.arange(len(frame.rx_data)) / plots["raw_sample_rate"] * 1e6
    plots["line_raw_i"].set_data(raw_time_us, frame.rx_data.real)
    plots["line_raw_q"].set_data(raw_time_us, frame.rx_data.imag)
    plots["line_raw_abs"].set_data(raw_time_us, np.abs(frame.rx_data))
    raw_limit = max(float(np.max(np.abs(frame.rx_data))), 1.0)
    plots["ax_raw"].set_ylim(-1.1 * raw_limit, 1.1 * raw_limit)
    plots["ax_raw"].set_xlim(0, len(frame.rx_data) / plots["raw_sample_rate"] * 1e6)
    plots["ax_raw"].set_title(
        f'Raw dechirped I/Q - Frame {frame_count} (assuming first trigger at t = 0)',
        fontsize=13, fontweight='bold'
    )
    plots["line_mag"].set_data(frame.freqs_khz, frame.magnitude_db)
    plots["ax1"].set_ylim([np.max(frame.magnitude_db)-60, np.max(frame.magnitude_db)+5])
    plots["ax1"].set_title(f'FFT Spectrum - Frame {frame_count}', fontsize=13, fontweight='bold')

    peak_mag = frame.magnitude_db[frame.target_bin]
    plots["peak_marker1"].set_data([frame.peak_freq], [peak_mag])

    plots["line_phase"].set_data(frame.freqs_khz, frame.phase_diff_deg)
    plots["ax2"].set_title(f'Phase Difference - Frame {frame_count}', fontsize=13, fontweight='bold')

    peak_phase = frame.phase_diff_deg[frame.target_bin]
    plots["peak_marker2"].set_data([frame.peak_freq], [peak_phase])

    # Keep the labels beside their markers and directed into the plot.
    for axis, label, peak_y, label_text in (
        (plots["ax1"], plots["peak_range_label"], peak_mag, f'{frame.range_bins_m[frame.target_bin]-CABLE_CALIBARTION_M:.2f} m'),
        (plots["ax2"], plots["peak_velocity_label"], peak_phase, f'{frame.velocity:+.2f} m/s'),
    ):
        label.xy = (frame.peak_freq, peak_y)
        label.set_text(label_text)
        x_low, x_high = axis.get_xlim()
        y_low, y_high = axis.get_ylim()
        right_half = frame.peak_freq > (x_low + x_high) / 2
        upper_half = peak_y > (y_low + y_high) / 2
        label.set_position((-12 if right_half else 12, -12 if upper_half else 12))
        label.set_ha('right' if right_half else 'left')
        label.set_va('top' if upper_half else 'bottom')



# Play-driven capture -> process -> update, matching the range-Doppler demo.

# Detach the previous controller when rerunning this cell or switching demos.
if "play" in globals() and isinstance(play, widgets.Play):
    play.playing = False
    play.disabled = True
    play.unobserve_all(name="value")
if "phase_plots" in globals():
    plt.close(phase_plots["fig"])

PHASE_CALIBRATION_DEG = 59
CABLE_CALIBARTION_M = 1.71
phase_sample_rate = float(sdr.sample_rate)
phase_plots = create_plots(phase_sample_rate, sdr.rx_buffer_size, tdd.frame_length_ms, config)
phase_frame_count = 0
phase_needs_warmup = True

out_info = widgets.Output(
    layout=widgets.Layout(border='1px solid #ccc', padding='10px', height='120px', overflow='auto'))

plot_widget = widgets.Image(
    value=render_plot(phase_plots['canvas']), format='png',
    layout=widgets.Layout(width='100%', border='1px solid #ccc'))

play = widgets.Play(value=0, min=0, max=1000, step=1, interval=500, description="Press play")

target_info = widgets.Output(layout=widgets.Layout(width='100%'))

with target_info:
    clear_output(wait=True)
    display(HTML(
        '<p style="color:#52637a;padding:12px;">Press Play to show target measurements.</p>'
    ))


def update_phase_demo(change):
    """Capture and display one frame when the Play value advances."""
    global phase_frame_count, phase_needs_warmup
    if play.disabled or change['new'] <= change['old']:
        return  # Reset/rewind does not capture data.
    try:
        if phase_needs_warmup:
            capture_data(phaser, sdr)
            phase_needs_warmup = False
        raw_data = capture_data(phaser, sdr)
        frame = process_frame(raw_data, phase_sample_rate, config, PHASE_CALIBRATION_DEG)
        phase_frame_count += 1
        update_plots(phase_plots, frame, phase_frame_count)
        plot_widget.value = render_plot(phase_plots['canvas'])
        with target_info:
            clear_output(wait=True)
            display(HTML(format_target_info(frame, phase_frame_count)))
    except (Exception, KeyboardInterrupt) as error:
        play.playing = False
        play.disabled = True
        with out_info:
            clear_output(wait=True)
            print(f'Capture stopped: {type(error).__name__}: {error}\n'
                                   'Fix the problem, then rerun this cell to enable Play again.')


play.observe(update_phase_demo, names='value')
layout = widgets.VBox([out_info, play, plot_widget, target_info])
# Display setup information once; plots and target measurements update each frame.
with out_info:
    clear_output(wait=True)
    display(HTML(
        f'<b>FMCW phase demo &middot; {config.num_chirps} chirps per capture</b><br>'
        f'Sample rate: {phase_sample_rate/1e6:.3f} MS/s &middot; RX buffer: {sdr.rx_buffer_size:,} samples<br>'
        f'Ramp: {config.ramp_time_us} us &middot; PRI: {float(tdd.frame_length_ms)*1e3:.0f} us '
        f'&middot; bandwidth: {config.chirp_BW/1e6:.0f} MHz<br>'
        f'Phase correction: {PHASE_CALIBRATION_DEG:+.1f}&deg; '
        f'&middot; range-label correction: {CABLE_CALIBARTION_M:.2f} m<br>'
        'Press Play to start; Pause to stop updates. The first tick includes a warm-up capture.'
    ))
display(layout)


## Improving Velocity Measurements with Multiple Chirps

We now have the ability to measure both distance and velocity using FMCW radar.

Using only two chirps, velocity is calculated from the phase difference between consecutive chirps. This works well, but the measurement can be sensitive to noise because it is based on a single phase comparison.

One way to improve noise tolerance is to increase the number of chirps and average the results. However, something even more interesting happens when we collect many chirps.

Let's assume the phase of a target changes by 10° per chirp due to its motion:

Chirp 1 :   0°
Chirp 2 :  10°
Chirp 3 :  20°
Chirp 4 :  30°
Chirp 5 :  40°
...

If we look at a single range bin over many chirps, the phase is steadily rotating. In complex form, this appears as a sampled sinusoid:

$
\Large
z[m] = A e^{j\phi_m}
$

where:

$
\Large
\phi_m = \phi_0 + 2\pi f_D\,m\,T_{\mathrm{PRI}}, \qquad m = 0,1,\ldots,M-1
$


where $m$ is the chirp index, $M$ is the number of chirps, $\phi_0$ is the
initial phase, and $f_D$ is the signed Doppler frequency in the complex-signal
convention defined above. The fast-time sample index will be denoted by $n$.
The phase difference is $\phi_{m+1}-\phi_m=2\pi f_DT_{\mathrm{PRI}}$;
the accumulated phase includes the chirp index $m$.

This means that for each range bin we have a signal that varies over time from chirp to chirp. This dimension is called slow time.

We can therefore perform a second FFT across the chirps, known as a slow-time FFT or Doppler FFT.

<div style="text-align: center;">
     <img src="resources/slow-time-fft.png" alt="slow time explanation" width="900">
</div>


# 3: Range-Doppler Processing

By combining the range information from the fast-time FFT with the velocity information from the slow-time FFT, we can create what is known as a **Range-Doppler Map**.

## Range-Doppler Map

A Range-Doppler map is a two-dimensional representation of the radar scene, showing both the distance and velocity of detected targets.

- **X-axis:** Radial Velocity
- **Y-axis:** Target Range
- **Colour:** Reflected Signal Strength

Each bright spot represents a target at a specific distance and speed.

Targets that appear at the same range can now be separated by their velocity, while targets travelling at similar speeds can be distinguished by their range.

The Range-Doppler map is one of the most important outputs of an FMCW radar, providing a clear view of **where targets are located and how fast they are moving.**


### Range Doppler Demo

This example updates the existing `config` with **128 chirps** and a **524,288-sample buffer**,
then calls the earlier SDR, PLL, TDD and TX setup functions to apply those values.  

Other configuration values, including gains and chirp settings, are reused.

Only the slow-time processing and new plot layout
are added here. The shared `process_chirps` function has an optional
`return_complex=True` argument to retain the phase needed for Doppler.

These settings remain active
after stopping; to return to two chirps, rerun the configuration-definition and
hardware-setup cells first.

- Each **row** contains one chirp; each **column** is the same fast-time sample
  position within successive chirps. The configured settling interval is excluded.
- The **fast-time FFT** runs across columns (`axis=1`) to separate range bins.
- The **slow-time FFT** runs down rows (`axis=0`) on the **complex** range FFT
  values to separate Doppler frequencies. Magnitude is taken afterwards.
- The raw plot shows the first four chirps for readability; both FFT displays use
  all 128. The range profile averages power across chirps. The map retains static
  clutter and shows intensity relative to the strongest displayed bin each frame.

At a 600 microsecond PRI, 128 chirps give a 76.8 ms CPI and approximately
0.197 m/s Doppler-bin spacing. Windowing broadens target peaks. The map uses velocity
horizontally and range vertically, with zero Doppler in the middle.

`RD_PHASE_CORRECTION_DEG` defaults to the previous demo's 59-degree correction.
It is applied as a phase progression across chirps. Confirm that a stationary
reference is near zero velocity with this larger burst; set it to zero to inspect
raw Doppler. `RD_MAX_RANGE_M` defaults to 15 m. Range still includes cable and
hardware delay; no wall-based range calibration has been assumed.


Run the hardware configuration cell, then the display cell. Press **Play** to
capture and update the plots; use **Pause** to stop requesting updates. Each tick
captures one 128-chirp burst (the first also discards a warm-up capture). The
500 ms Play interval controls display requests, not the chirp PRI or velocity
calculation. If processing takes longer, updates can lag behind the control.
No continuous loop or `await` is needed in the display cell. Run the earlier demo cell to define its shared helpers; it returns immediately.
Pause its Play widget before changing the hardware setup for this demo.


In [ ]:
config.num_chirps = 128
config.sdr_buf_size = 2**19  # 524,288 samples for the 128-chirp burst.

# Apply the updated config using the same hardware functions as the earlier demo.
# configure_sdr calculates the sample rate from buffer size / complete burst time.
hardware_configuration(connect = False)

In [ ]:
# Range-Doppler demo: 128 chirps, fast-time FFT followed by slow-time FFT.
# Run the previous demo cell once to define its helpers; pause it before configuring this demo.
# Reuse capture_data, combine_rx_channels, extract_chirps, process_chirps and render_plot.
# Run the preceding 128-chirp hardware configuration cell, then run this cell and press Play.
from IPython.display import HTML

# Detach the previous Play widget when this cell is run again.
if "play" in globals() and isinstance(play, widgets.Play):
    play.playing = False
    play.disabled = True
    play.unobserve_all(name="value")
if "rd_plots" in globals() and rd_plots is not None:
    plt.close(rd_plots["fig"])

# Parameters for this demo (other RadarConfig values remain as configured).
RD_MAX_RANGE_M = 15             # Apparent range: cable/path delay is not removed.
RD_PHASE_CORRECTION_DEG = 59.0   # Added per chirp interval; set to 0 for raw Doppler.
RD_RAW_CHIRPS_TO_SHOW = 2        # Display two chirps; process all 128.
RD_DYNAMIC_RANGE_DB = 60

def process_range_doppler(chirps, sample_rate, pri_s, ramp_s, bandwidth_hz,
                          if_hz, carrier_hz, phase_correction_deg, max_range_m):
    """
    Keep complex range FFT values so their slow-time phase contains velocity.
    """
    
    num_chirps, num_samples = chirps.shape
    wavelength = 3e8 / carrier_hz
    slope = bandwidth_hz / ramp_s

    # 1. FAST TIME: FFT across columns (samples within each chirp).
    fast_window = np.blackman(num_samples)
    range_fft = process_chirps(chirps, return_complex=True) / fast_window.sum()
    frequencies = np.fft.fftfreq(num_samples, d=1 / sample_rate)
    ranges = (frequencies - if_hz) * 3e8 / (2 * slope)
    
    # Select the positive-range side of the IF, matching the previous display.
    visible = (frequencies >= if_hz) & (ranges <= max_range_m)
    range_bins = ranges[visible]
    
    if len(range_bins) < 2:
        raise ValueError("Not enough range bins in the display interval; check sample rate and IF.")
    
    range_fft = range_fft[:, visible]
    range_power = np.mean(np.abs(range_fft) ** 2, axis=0)
    range_db = 10 * np.log10(range_power + 1e-12)

    # Apply the measured inter-chirp correction as a phase progression, not
    # a constant rotation of every row (which would not shift Doppler).
    correction = np.exp(1j * np.arange(num_chirps) * np.deg2rad(phase_correction_deg))
    range_fft = range_fft * correction[:, None]

    # 2. SLOW TIME: FFT down rows (successive chirps at each range bin).
    # Do not take abs() before this FFT: it would discard Doppler phase.
    slow_window = np.hanning(num_chirps)
    doppler_fft = np.fft.fft(range_fft * slow_window[:, None], axis=0) / slow_window.sum()
    doppler_fft = np.fft.fftshift(doppler_fft, axes=0)
    doppler_hz = np.fft.fftshift(np.fft.fftfreq(num_chirps, d=pri_s))
    velocity_bins = doppler_hz * wavelength / 2
    map_db = 20 * np.log10(np.abs(doppler_fft) + 1e-12)
    map_db -= map_db.max()  # Colour is dB relative to the strongest bin in this CPI.
    
    return range_bins, velocity_bins, range_db, map_db



def create_rd_plots(sample_rate, pri_s, ramp_s, skip_fraction, range_bins, velocity_bins):
    """
    Create a raw-buffer plot, averaged range FFT, and range-Doppler map.
    """
    fig = Figure(figsize=(14, 15), dpi=90)
    canvas = FigureCanvasAgg(fig)
    
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 2])
    ax_time = fig.add_subplot(gs[0, 0])
    ax_range = fig.add_subplot(gs[0, 1])
    ax_rd = fig.add_subplot(gs[1, :])
   
    line_i, = ax_time.plot([], [], linewidth=0.6, alpha=0.7, label='I (RX0 + RX1)')
    line_q, = ax_time.plot([], [], linewidth=0.6, alpha=0.7, label='Q (RX0 + RX1)')
    line_abs, = ax_time.plot([], [], color='black', linewidth=0.9, label='Magnitude')
    
    for chirp in range(min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps)):
        start_us = chirp * pri_s * 1e6
        end_us = start_us + ramp_s * 1e6
        ax_time.axvline(start_us, color='tab:green', linestyle='--',
                        label='Expected ramp start' if chirp == 0 else None)
        ax_time.axvline(end_us, color='tab:red', linestyle='--',
                        label='Expected ramp end' if chirp == 0 else None)
        ax_time.axvspan(start_us, start_us + skip_fraction * ramp_s * 1e6,
                        color='gold', alpha=0.2, label='Excluded settling' if chirp == 0 else None)
        ax_time.axvspan(end_us, start_us + pri_s * 1e6, color='grey', alpha=0.15,
                        label='Padding' if chirp == 0 else None)
        
    ax_time.set_xlabel('Time from first RX sample (us)')
    ax_time.set_ylabel('Raw amplitude (ADC counts)')
    ax_time.set_xlim(0, min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps) * pri_s * 1e6)
    ax_time.set_title('Raw dechirped I/Q - expected first trigger at t = 0')
    ax_time.legend(loc='lower left', ncol=4, fontsize=8)
    ax_time.grid(True, alpha=0.3)

    line_range, = ax_range.plot([], [], color='tab:blue', linewidth=1.2)
    ax_range.set(xlabel='Apparent range (m)', ylabel='Mean power (dB re 1 ADC count squared)',
                 xlim=(0, RD_MAX_RANGE_M), title=f'Fast-time FFT - power averaged over {config.num_chirps} chirps')
    ax_range.grid(True, alpha=0.3)

    # imshow takes pixel edges; the FFT arrays contain pixel centres.
    dr = range_bins[1] - range_bins[0]
    dv = velocity_bins[1] - velocity_bins[0]
    
    extent = [velocity_bins[0] - dv/2, velocity_bins[-1] + dv/2,
              range_bins[0] - dr/2, range_bins[-1] + dr/2]
   
    image = ax_rd.imshow(np.full((len(range_bins), len(velocity_bins)), -RD_DYNAMIC_RANGE_DB),
                         origin='lower', aspect='auto', interpolation='nearest', extent=extent,
                         cmap='inferno', vmin=-RD_DYNAMIC_RANGE_DB, vmax=0)
    
    ax_rd.set(xlabel='Radial velocity (m/s)', ylabel='Apparent range (m)',
              xlim=(-5, 5), ylim=(0, RD_MAX_RANGE_M), title='Range-Doppler map - fast-time then slow-time FFT')
    
    ax_rd.axvline(0, color='white', linewidth=0.7, alpha=0.5)
    
    
    fig.tight_layout()
    
    return dict(fig=fig, canvas=canvas, ax_time=ax_time, ax_range=ax_range, ax_rd=ax_rd,
                line_i=line_i, line_q=line_q, line_abs=line_abs, line_range=line_range,
                image=image)


def update_rd_plots(plots, rx_data, sample_rate, pri_s, range_bins, range_db, map_db, frame_count):
    """
    Update the same three plots for each captured CPI.
    """
    shown = min(len(rx_data), round(min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps) * pri_s * sample_rate))
    raw = rx_data[:shown]
    
    times_us = np.arange(shown) / sample_rate * 1e6
    
    plots['line_i'].set_data(times_us, raw.real)
    plots['line_q'].set_data(times_us, raw.imag)
    plots['line_abs'].set_data(times_us, np.abs(raw))
   
    limit = max(float(np.abs(raw).max()), 1.0)
    
    plots['ax_time'].set_ylim(-1.1 * limit, 1.1 * limit)
    plots['ax_time'].set_title(
        f'Raw I/Q - first {min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps)} of {config.num_chirps} chirps '
        f'- frame {frame_count} (assuming trigger at t = 0)')
    
    plots['line_range'].set_data(range_bins, range_db)
    plots['ax_range'].set_ylim(range_db.max() - RD_DYNAMIC_RANGE_DB, range_db.max() + 5)
    plots['ax_rd'].set_title(f'Range-Doppler map - {config.num_chirps} chirps - frame {frame_count}')
    plots['image'].set_data(map_db.T)  # Rows are range; columns are velocity.


# Read timing once, after running the hardware configuration cell above.
rd_sample_rate  = float(sdr.sample_rate)
rd_pri_s        = float(tdd.frame_length_ms) * 1e-3
rd_ramp_s       = float(phaser.freq_dev_time) * 1e-6
rd_bandwidth_hz = 4 * float(phaser.freq_dev_range)

if config.num_chirps < 3 or int(tdd.burst_count) != config.num_chirps:
    raise ValueError("Run the preceding range-Doppler hardware configuration cell first.")

if not 0 <= config.begin_offset_fraction < 1:
    raise ValueError("Settling fraction must be between 0 and 1.")

rd_skip_samples = int(config.begin_offset_fraction * config.ramp_time_us * 1e-6 * rd_sample_rate)

if int(config.ramp_time_us * 1e-6 * rd_sample_rate) - rd_skip_samples < 3:
    raise ValueError("Too few usable samples per ramp for the range FFT.")

# One layout, one image widget, and one callback for each Play value change.
out_info = widgets.Output(
    layout=widgets.Layout(border='1px solid #ccc', padding='10px'))

plot_widget = widgets.Image(
    format='png', layout=widgets.Layout(width='100%', border='1px solid #ccc'))

play = widgets.Play(value=0, min=0, max=1000, step=1, interval=100, description="Press play", align = "centre")

rd_plots        = None
rd_frame_count  = 0
rd_needs_warmup = True


def update(change):
    """Capture one CPI, process it, and refresh the displayed image."""
    global rd_plots, rd_frame_count, rd_needs_warmup
    
    if play.disabled or change['new'] <= change['old']:
        return  # Reset/rewind does not acquire a frame.
    try:
        if rd_needs_warmup:
            capture_data(phaser, sdr)  # Discard the first capture once, on first Play.
            rd_needs_warmup = False
        raw_data = capture_data(phaser, sdr)
        rx_data = combine_rx_channels(raw_data)
        chirps = extract_chirps(rx_data, config.num_chirps, rd_sample_rate, config)
        chirps = chirps[:, rd_skip_samples:]
        ranges, velocities, range_db, map_db = process_range_doppler(
            chirps, rd_sample_rate, rd_pri_s, rd_ramp_s, rd_bandwidth_hz,
            config.signal_freq, config.output_freq, RD_PHASE_CORRECTION_DEG, RD_MAX_RANGE_M)
        if rd_plots is None:
            rd_plots = create_rd_plots(rd_sample_rate, rd_pri_s, rd_ramp_s,
                                       config.begin_offset_fraction, ranges, velocities)
        rd_frame_count += 1
        update_rd_plots(rd_plots, rx_data, rd_sample_rate, rd_pri_s,
                        ranges, range_db, map_db, rd_frame_count)
        plot_widget.value = render_plot(rd_plots['canvas'])

    except (Exception, KeyboardInterrupt) as error:
        # Callback errors must be shown in the Output widget, not lost in kernel logs.
        play.playing = False
        play.disabled = True
        with out_info:
            clear_output(wait=True)
            print(f'Capture stopped: {type(error).__name__}: {error}\n'
                                   'Fix the problem, then rerun this cell to enable Play again.')


play.observe(update, names='value')

layout = widgets.VBox([out_info, play, plot_widget])

# Setup information is displayed once; only the plot changes on each frame.
with out_info:
    clear_output(wait=True)
    display(HTML(
        f'<b>Range-Doppler setup &middot; {config.num_chirps} chirps per capture</b><br>'
        f'Sample rate: {rd_sample_rate/1e6:.3f} MS/s &middot; '
        f'RX buffer: {sdr.rx_buffer_size:,} samples<br>'
        f'Ramp: {rd_ramp_s * 1e6:.0f} us &middot; PRI: {rd_pri_s * 1e6:.0f} us '
        f'&middot; CPI: {config.num_chirps * rd_pri_s * 1e3:.1f} ms<br>'
        f'Doppler bin: {(3e8 / config.output_freq) / (2 * config.num_chirps * rd_pri_s):.3f} m/s '
        f'&middot; phase correction: {RD_PHASE_CORRECTION_DEG:+.1f}&deg;/chirp<br>'
        f'Range display: 0&ndash;{RD_MAX_RANGE_M:g} m (includes cable/path delay). '
        'Press Play to start; Pause to stop updates.'
    ))

display(layout)


# 4: Building the CPI Data Cube

This section follows the **same 128-chirp configuration as the live demo**.
A coherent processing interval (CPI) contains $M$ chirps with fixed ramp settings,
a fixed interval between chirp starts, and consistent inter-chirp phase.

## Timing and Buffer Size

| Parameter | Demo value |
| --- | --- |
| Chirps per CPI, $M$ | 128 |
| Ramp duration, $T_{\mathrm{ramp}}$ | 500 microseconds |
| Padding, $T_{\mathrm{pad}}$ | 100 microseconds |
| Chirp-start interval, $T_{\mathrm{PRI}}$ | 600 microseconds |
| Chirp bandwidth, $B$ | 500 MHz |
| Chirp slope, $S=B/T_{\mathrm{ramp}}$ | $10^{12}$ Hz/s |
| Receive buffer size | 524,288 complex samples **per RX channel** |

The complete burst, including padding after each ramp, occupies

$$
T_{\mathrm{CPI}} = M T_{\mathrm{PRI}}
= 128\times600~\mu\mathrm{s} = 76.8~\mathrm{ms}.
$$

The setup function requests the sample rate from the buffer size and total burst time:

$$
F_s = \left\lfloor\frac{524288}{0.0768}\right\rfloor
= 6{,}826{,}666~\mathrm{samples/s}.
$$

The code reads back `sdr.sample_rate`; that actual value is used for processing.
At this nominal rate, one PRI contains approximately 4096 samples. Thus

$$
128\times4096=524288
$$

samples cover the whole burst. **Padding consumes receive-buffer samples even
though those samples are not passed to the range FFT.** For independent timing
and sample-rate choices, budget at least $\lceil F_s M T_{\mathrm{PRI}}\rceil$
samples for a complete burst starting at sample zero; include any additional
capture offset as well.

Both RX channels have this many complex samples. Enabling two channels doubles
the combined data volume, not the value of `sdr.rx_buffer_size`, which is per channel.

## From the Raw Buffer to the Chirp Matrix

There are three different sample counts:

1. **Samples per PRI:** approximately 4096, including the ramp and padding.
2. **Samples per ramp:** `int(Fs * 500e-6)` = 3413 at the nominal rate.
3. **Samples used by the range FFT:** the ramp samples remaining after the
   configured settling interval is removed.

With the current `begin_offset_fraction = 0.05`, the excluded interval is
25 microseconds. The code removes

```python
skip_samples = int(0.05 * 500e-6 * 6_826_666)  # 170
N = 3413 - skip_samples                       # 3243
M = 128
```

These counts describe the current configuration; recompute them from the actual
sample-rate readback and config if settings change. `N` is not an independently
chosen 1024-point FFT in this demo.

The two receiver channels are first combined by `combine_rx_channels()`. After
extraction and settling removal, the matrix used by the FFTs has shape

```python
chirps.shape == (128, 3243)  # (slow time, fast time), at the nominal sample rate
```

Each **row** is one chirp. Each **column** is the same fast-time position across
successive chirps. The range FFT runs across rows (`axis=1`); the Doppler FFT
runs down columns (`axis=0`). Keeping the channels separate would instead give
a cube of shape `(128, 3243, 2)`, but that is not what these demos process.

## What Sets the Range Axis?

The fast-time FFT bin spacing is

$$
\Delta f = \frac{F_s}{N} \approx 2105.05~\mathrm{Hz}, \qquad
\Delta R_{\mathrm{bin}} = \frac{c\,\Delta f}{2S}
\approx 0.316~\mathrm{m}.
$$

The ideal resolution using the complete 500 MHz ramp bandwidth is
$c/(2B)=0.300$ m. Excluding the start of the ramp shortens the observation and
reduces its effective bandwidth. The Blackman window also broadens target peaks,
so bin spacing is not a guarantee that two targets that close can be separated.
Zero-padding can interpolate the displayed spectrum but does not add physical
range resolution.

The range-Doppler demo uses the sideband above the 100 kHz transmit IF offset:

$$
f_{\mathrm{range}} = S\frac{2R}{c}, \qquad
f_{\mathrm{sampled}} \approx f_{\mathrm{IF}} + f_{\mathrm{range}}.
$$

Ignoring the much smaller motion-dependent frequency contribution, a 10 m
apparent range gives about 66.7 kHz of range beat frequency, or **166.7 kHz in the
sampled spectrum** after including the IF. Sampling and filtering must accommodate
the sampled frequency, not just the range beat. For the selected positive-frequency
representation, it must lie below $F_s/2$ and within the usable receiver bandwidth.
The displayed 0-15 m interval is a plotting choice, not the hardware range limit.
Cable and internal path delays remain included in the apparent-range axis.

## What Sets the Velocity Axis?

The slow-time sampling rate is the PRF, not the ADC sample rate:

$$
\mathrm{PRF} = \frac{1}{T_{\mathrm{PRI}}}
= \frac{1}{600~\mu\mathrm{s}} \approx 1666.67~\mathrm{Hz}.
$$

For $M=128$, the Doppler FFT bin spacing is $1/(M T_{\mathrm{PRI}})$, approximately
13.02 Hz. At 9.9 GHz, this corresponds to **0.197 m/s per velocity bin**. Increasing
$M$ at the same PRI gives finer Doppler bins and a longer CPI. It does not change
the unambiguous velocity limit, which depends on PRI.

The first and last chirp starts are $(M-1)T_{\mathrm{PRI}}$ apart; the complete
buffer spans $M T_{\mathrm{PRI}}$, including the last chirp and its padding.
The Play interval determines when another burst is requested, not the spacing
between the 128 chirps within that burst.


## Velocity Bin Spacing and Resolution

For $M$ chirps spaced by $T_{\mathrm{PRI}}$, the Doppler FFT samples frequency in steps of

$$
\Delta f_D = \frac{1}{M T_{\mathrm{PRI}}}, \qquad
\Delta v_{\mathrm{bin}} = \frac{\lambda}{2M T_{\mathrm{PRI}}}.
$$

Here $\lambda=c/f_c$, and $T_{\mathrm{PRI}}$ is the interval between chirp starts,
**including padding**. For $f_c=9.9$ GHz, $M=128$ and $T_{\mathrm{PRI}}=600$ microseconds:

$$
\lambda \approx 0.0303~\mathrm{m}, \qquad
\Delta v_{\mathrm{bin}} \approx 0.197~\mathrm{m/s}.
$$

More chirps give finer Doppler bins at the cost of a longer CPI. The slow-time
Hann window broadens peaks, so actual two-target velocity resolution is coarser
than the native bin spacing. Zero-padding does not improve that physical resolution.

## Maximum Unambiguous Velocity

Sampling once per PRI gives a signed Doppler interval
$-\mathrm{PRF}/2 \le f_D < \mathrm{PRF}/2$. Therefore

$$
v_{\max} = \frac{\lambda}{4T_{\mathrm{PRI}}}
= \frac{\lambda\,\mathrm{PRF}}{4}
\approx 12.63~\mathrm{m/s}.
$$

The corresponding signed velocity interval is $-v_{\max} \le v_r < v_{\max}$.
Velocities outside it alias into that interval. The plotted -5 to +5 m/s window
zooms into walking speeds; it does not change the underlying ambiguity limit.
The theoretical convention is positive toward the radar; check the measured
I/Q phase sign against known motion before assigning approaching/receding labels.


# 5: Summary and Next Steps

## What We've Accomplished

In this notebook, we:
1. ✓ Understood fast-time vs slow-time dimensions
2. ✓ Captured coherent multi-chirp CPIs
3. ✓ Implemented the 2D FFT (Range-Doppler processing)
4. ✓ Generated Range-Doppler Maps


## Key Insights

- **2D FFT** provides simultaneous range-velocity measurement
- **More chirps** → better velocity resolution (but longer CPI)
- **Range-velocity coupling** exists but can be mitigated

## Current Capabilities

We can now:
- Detect multiple targets
- Measure range and velocity


## Next Notebook: Beamforming Integration

In the next notebook (`5_FMCW_Beamforming_Integration.ipynb`), we'll:
- Add **angle estimation** using the phased array
- Implement **3D tracking**: Range + Velocity + Angle
- Use beam steering to focus on specific targets
- Create **Range-Angle-Doppler cubes**
- Demonstrate full RADAR tracking capabilities

This brings together everything: beamforming + FMCW + Doppler processing!

# Appendix A: 2D FFT Mathematics

## Discrete 2D FFT

For data matrix $x[m, n]$ of size $M \times N$:

$$
X[k, l] = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} x[m, n] \cdot e^{-j2\pi\left(\frac{km}{M} + \frac{ln}{N}\right)}
$$

## Separable Transform

The 2D FFT can be computed as two sequential 1D FFTs:

1. $Y[m, l] = \text{FFT}_N\{x[m, n]\}$ (FFT along each row)
2. $X[k, l] = \text{FFT}_M\{Y[m, l]\}$ (FFT along each column)

## Computational Complexity

- 1D FFT: $O(N \log N)$
- 2D FFT: $O(MN(\log M + \log N))$

The current example uses $M=128$ and approximately $N=3243$ after settling
removal. The expression above describes how FFT work scales, rather than an
exact operation count. Acquisition, FFT processing and PNG rendering all
contribute to the time required for each Play update.

# Appendix B: Doppler Ambiguity Resolution

## PRF Selection Trade-offs

| PRF | Advantage | Disadvantage |
|-----|-----------|-------------|
| High PRF | Large $v_{\text{max}}$ | Small $R_{\text{max}}$ |
| Low PRF | Large $R_{\text{max}}$ | Small $v_{\text{max}}$ |
| Medium PRF | Balanced | Ambiguities in both |

## Multiple PRF Technique

Use two or more PRFs and resolve ambiguities using Chinese Remainder Theorem:
1. Capture CPI at PRF₁
2. Capture CPI at PRF₂
3. Resolve velocity ambiguity algebraically

Extended unambiguous velocity:
$$
v_{\text{max,extended}} = v_{\text{max,1}} \times v_{\text{max,2}} / \gcd(v_{\text{max,1}}, v_{\text{max,2}})
$$